# 02 — Model Training: Credit Risk GBM

Train the same LightGBM classifier and deterministic early-stopping split used by the production pipeline.  
Evaluate on the untouched temporal test set and the early-stopping carve-out. Compute AUC, KS, Gini, calibration, and decile analysis without writing model artifacts.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve


def find_repo_root():
    """Locate the repository whether Jupyter starts here or in notebooks/."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pipeline").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the GBM_cs repository")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from pipeline.train import load_gold_data, train_model  # noqa: E402

plt.style.use("seaborn-v0_8-whitegrid")

X_train, y_train, X_val, y_val, X_test, y_test, feature_cols = load_gold_data()
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

## 1. Train Model

In [ ]:
# Reuse the production training implementation so this notebook cannot drift.
# X_es and y_es are the deterministic carve-out used for early stopping.
model, params, X_es, y_es = train_model(X_train, y_train, X_val, y_val)
print(f"Best iteration: {model.best_iteration_}")
print(f"Training parameters: {params}")

## 2. Evaluation — ROC, Precision-Recall, Calibration

In [ ]:
def compute_ks(y_true, y_score):
    fpr, tpr, _ = roc_curve(y_true, y_score)
    return float(np.max(tpr - fpr))


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for X, y, name, color in [
    (X_train, y_train, "Train", "#2ecc71"),
    (X_es, y_es, "Early stop", "#f39c12"),
    (X_test, y_test, "Test", "#e74c3c"),
]:
    scores = model.predict_proba(X)[:, 1]
    auc = roc_auc_score(y, scores)
    ks = compute_ks(y, scores)

    fpr, tpr, _ = roc_curve(y, scores)
    axes[0].plot(fpr, tpr, color=color, label=f"{name} (AUC={auc:.3f})")

    precision, recall, _ = precision_recall_curve(y, scores)
    axes[1].plot(recall, precision, color=color, label=name)

    prob_true, prob_pred = calibration_curve(y, scores, n_bins=10)
    axes[2].plot(prob_pred, prob_true, "s-", color=color, label=name)

    print(f"{name:10s} | AUC={auc:.4f}  KS={ks:.4f}  Gini={2 * auc - 1:.4f}")

axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC Curve")
axes[0].legend()

axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve")
axes[1].legend()

axes[2].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[2].set_xlabel("Predicted Probability")
axes[2].set_ylabel("Actual Probability")
axes[2].set_title("Calibration Curve")
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Decile Analysis — Risk Rank Ordering

In [ ]:
# Decile analysis on the untouched temporal test set.
test_scores = model.predict_proba(X_test)[:, 1]
decile_df = pd.DataFrame({"score": test_scores, "default": y_test.to_numpy()})
# Rank first so tied model scores cannot make qcut drop or reject bins.
decile_df["decile"] = pd.qcut(
    decile_df["score"].rank(method="first"), 10, labels=range(1, 11)
)

decile_stats = decile_df.groupby("decile", observed=True).agg(
    count=("default", "size"),
    n_defaults=("default", "sum"),
    default_rate=("default", "mean"),
    avg_score=("score", "mean"),
    min_score=("score", "min"),
    max_score=("score", "max"),
).reset_index()

decile_stats["cumulative_defaults"] = decile_stats["n_defaults"].cumsum()
decile_stats["capture_rate"] = (
    decile_stats["cumulative_defaults"] / decile_stats["n_defaults"].sum()
)

print(decile_stats.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(12, 6))
ax1.bar(
    decile_stats["decile"].astype(str),
    decile_stats["default_rate"],
    color=plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 10)),
    edgecolor="black",
)
ax1.set_xlabel("Score Decile (1=lowest risk)")
ax1.set_ylabel("Default Rate", color="black")
ax1.set_title("Test Set: Default Rate & Cumulative Capture by Score Decile")

ax2 = ax1.twinx()
ax2.plot(
    decile_stats["decile"].astype(str),
    decile_stats["capture_rate"],
    "ko-",
    linewidth=2,
    markersize=8,
    label="Cumulative Capture",
)
ax2.set_ylabel("Cumulative Default Capture Rate", color="black")
ax2.legend(loc="center right")

plt.tight_layout()
plt.show()

top_three_capture = (
    decile_stats.iloc[-3:]["n_defaults"].sum()
    / decile_stats["n_defaults"].sum()
)
print(f"\nTop 3 deciles capture {top_three_capture:.1%} of all defaults")